# Feature Engineering — Premier League 2025/26

This notebook prepares features that can be used for machine learning.

The main goal is to create features that represent information available before a match takes place.

We will focus on:

- Team historical performance
- Home and away form
- Historical goals
- Historical points
- Match context
- Identifying and avoiding data leakage

In [1]:
import pandas as pd
import numpy as np

## 1. Load the Cleaned Dataset

We load the cleaned dataset produced during Phase 2.

In [2]:
df = pd.read_csv("../data/processed/PremierLeague25_26_clean.csv")

df["Date"] = pd.to_datetime(df["Date"])

df = df.sort_values("Date").reset_index(drop=True)

df.head()

,Div,Date,Time,HomeTeam,AwayTeam,FTHG,FTAG,FTR,HTHG,HTAG,...,B365CAHH,B365CAHA,PCAHH,PCAHA,MaxCAHH,MaxCAHA,AvgCAHH,AvgCAHA,BFECAHH,BFECAHA
0,E0,2025-08-15,20:00,Liverpool,Bournemouth,4,2,H,1,0,...,2.03,1.78,2.07,1.85,2.03,1.88,1.94,1.76,2.14,1.86
1,E0,2025-08-16,12:30,Aston Villa,Newcastle,0,0,D,0,0,...,2.05,1.80,2.02,1.89,2.06,1.80,1.95,1.74,2.14,1.86
2,E0,2025-08-16,15:00,Brighton,Fulham,1,1,D,0,0,...,1.83,2.03,1.93,2.00,1.84,2.03,1.80,1.96,1.91,2.08
3,E0,2025-08-16,15:00,Sunderland,West Ham,3,0,H,0,0,...,1.95,1.90,1.97,1.95,1.95,1.94,1.86,1.78,2.02,1.97
4,E0,2025-08-16,15:00,Tottenham,Burnley,3,0,H,1,0,...,1.98,1.88,1.99,1.93,1.98,1.91,1.88,1.83,2.07,1.92


In [3]:
print("First match:", df["Date"].min())
print("Last match:", df["Date"].max())
df[["Date", "HomeTeam", "AwayTeam", "FTHG", "FTAG", "FTR"]].head(10)

First match: 2025-08-15 00:00:00
Last match: 2026-05-24 00:00:00


,Date,HomeTeam,AwayTeam,FTHG,FTAG,FTR
0,2025-08-15,Liverpool,Bournemouth,4,2,H
1,2025-08-16,Aston Villa,Newcastle,0,0,D
2,2025-08-16,Brighton,Fulham,1,1,D
3,2025-08-16,Sunderland,West Ham,3,0,H
4,2025-08-16,Tottenham,Burnley,3,0,H
5,2025-08-16,Wolves,Man City,0,4,A
6,2025-08-17,Chelsea,Crystal Palace,0,0,D
7,2025-08-17,Nott'm Forest,Brentford,3,1,H
8,2025-08-17,Man United,Arsenal,0,1,A
9,2025-08-18,Leeds,Everton,1,0,H


## 2. Identify Potential Target

The `FTR` column represents the full-time match result:

- `H` → Home win
- `D` → Draw
- `A` → Away win

This is a potential classification target.

We will formally finalize the ML problem later, but for feature engineering we will treat `FTR` as our candidate target.

In [4]:
df["FTR"].value_counts()

FTR
H    162
A    114
D    104
Name: count, dtype: int64

## 3. Identify Potentially Leaky Columns

Several columns describe events that happen during or after the match.

These must not be used as pre-match prediction features.

Examples include:

- FTHG
- FTAG
- FTR
- HTHG
- HTAG
- HTR
- HS
- AS
- HST
- AST
- HF
- AF
- HC
- AC
- HY
- AY
- HR
- AR

Using these columns to predict the same match result would allow the model to see information that would not be available before the match.

In [5]:
leaky_columns = [
    "FTHG", "FTAG", "FTR",
    "HTHG", "HTAG", "HTR",
    "HS", "AS",
    "HST", "AST",
    "HF", "AF",
    "HC", "AC",
    "HY", "AY",
    "HR", "AR"
]

print("Potentially leaky columns:")
print(leaky_columns)

Potentially leaky columns:
['FTHG', 'FTAG', 'FTR', 'HTHG', 'HTAG', 'HTR', 'HS', 'AS', 'HST', 'AST', 'HF', 'AF', 'HC', 'AC', 'HY', 'AY', 'HR', 'AR']


## 4. Historical Team Statistics

Instead of using the current match's statistics, we can calculate statistics from matches that happened previously.

For example, before Arsenal plays a match, we could know:

- Arsenal's previous average goals
- Arsenal's previous average goals conceded
- Arsenal's previous points
- Arsenal's recent form

These statistics can be used as pre-match features.

The key requirement is that the current match must not be included when calculating its own historical features.

In [6]:
def get_points(result):
    if result == "W":
        return 3
    elif result == "D":
        return 1
    else:
        return 0

## 5. Build Historical Team Records

We process matches chronologically.

For every team, we maintain statistics from matches that have already happened.

For each team we track:

- Matches played
- Goals scored
- Goals conceded
- Points

These statistics will be used to generate pre-match features.

In [7]:
team_stats = {}

feature_rows = []

for _, match in df.iterrows():

    home_team = match["HomeTeam"]
    away_team = match["AwayTeam"]

    # Initialize teams if they have not appeared before
    if home_team not in team_stats:
        team_stats[home_team] = {
            "matches": 0,
            "goals_for": 0,
            "goals_against": 0,
            "points": 0
        }

    if away_team not in team_stats:
        team_stats[away_team] = {
            "matches": 0,
            "goals_for": 0,
            "goals_against": 0,
            "points": 0
        }

    home_stats = team_stats[home_team]
    away_stats = team_stats[away_team]

    # Store the historical information BEFORE updating it
    feature_rows.append({
        "Date": match["Date"],
        "HomeTeam": home_team,
        "AwayTeam": away_team,

        "HomeMatchesBefore": home_stats["matches"],
        "HomeGoalsForBefore": home_stats["goals_for"],
        "HomeGoalsAgainstBefore": home_stats["goals_against"],
        "HomePointsBefore": home_stats["points"],

        "AwayMatchesBefore": away_stats["matches"],
        "AwayGoalsForBefore": away_stats["goals_for"],
        "AwayGoalsAgainstBefore": away_stats["goals_against"],
        "AwayPointsBefore": away_stats["points"],

        "FTR": match["FTR"]
    })

    # Update statistics AFTER the match
    home_goals = match["FTHG"]
    away_goals = match["FTAG"]

    home_stats["matches"] += 1
    away_stats["matches"] += 1

    home_stats["goals_for"] += home_goals
    home_stats["goals_against"] += away_goals

    away_stats["goals_for"] += away_goals
    away_stats["goals_against"] += home_goals

    if home_goals > away_goals:
        home_stats["points"] += 3

    elif home_goals < away_goals:
        away_stats["points"] += 3

    else:
        home_stats["points"] += 1
        away_stats["points"] += 1

In [8]:
features_df = pd.DataFrame(feature_rows)

features_df.head(10)

,Date,HomeTeam,AwayTeam,HomeMatchesBefore,HomeGoalsForBefore,HomeGoalsAgainstBefore,HomePointsBefore,AwayMatchesBefore,AwayGoalsForBefore,AwayGoalsAgainstBefore,AwayPointsBefore,FTR
0,2025-08-15,Liverpool,Bournemouth,0,0,0,0,0,0,0,0,H
1,2025-08-16,Aston Villa,Newcastle,0,0,0,0,0,0,0,0,D
2,2025-08-16,Brighton,Fulham,0,0,0,0,0,0,0,0,D
3,2025-08-16,Sunderland,West Ham,0,0,0,0,0,0,0,0,H
4,2025-08-16,Tottenham,Burnley,0,0,0,0,0,0,0,0,H
5,2025-08-16,Wolves,Man City,0,0,0,0,0,0,0,0,A
6,2025-08-17,Chelsea,Crystal Palace,0,0,0,0,0,0,0,0,D
7,2025-08-17,Nott'm Forest,Brentford,0,0,0,0,0,0,0,0,H
8,2025-08-17,Man United,Arsenal,0,0,0,0,0,0,0,0,A
9,2025-08-18,Leeds,Everton,0,0,0,0,0,0,0,0,H


In [9]:
features_df.shape

(380, 12)

## 6. Convert Historical Totals into Averages

Raw totals are affected by the number of matches a team has already played.

For example:

A team with 10 matches will naturally have more total goals than a team with only 3 matches.

Therefore, we can calculate averages using the number of matches played.

In [10]:
features_df["HomeAvgGoalsFor"] = np.where(
    features_df["HomeMatchesBefore"] > 0,
    features_df["HomeGoalsForBefore"] /
    features_df["HomeMatchesBefore"],
    0
)

features_df["HomeAvgGoalsAgainst"] = np.where(
    features_df["HomeMatchesBefore"] > 0,
    features_df["HomeGoalsAgainstBefore"] /
    features_df["HomeMatchesBefore"],
    0
)

features_df["AwayAvgGoalsFor"] = np.where(
    features_df["AwayMatchesBefore"] > 0,
    features_df["AwayGoalsForBefore"] /
    features_df["AwayMatchesBefore"],
    0
)

features_df["AwayAvgGoalsAgainst"] = np.where(
    features_df["AwayMatchesBefore"] > 0,
    features_df["AwayGoalsAgainstBefore"] /
    features_df["AwayMatchesBefore"],
    0
)

## 7. Historical Points Per Match

Points per match provides a measure of historical team performance.

It is calculated as:

Points Per Match = Total Historical Points / Matches Played

Again, these values only use matches that occurred before the current match.

In [11]:
features_df["HomePointsPerMatch"] = np.where(
    features_df["HomeMatchesBefore"] > 0,
    features_df["HomePointsBefore"] /
    features_df["HomeMatchesBefore"],
    0
)

features_df["AwayPointsPerMatch"] = np.where(
    features_df["AwayMatchesBefore"] > 0,
    features_df["AwayPointsBefore"] /
    features_df["AwayMatchesBefore"],
    0
)

## 8. Difference-Based Features

Instead of giving the model two separate values, we can also create comparison features.

For example:

Home attacking strength − Away attacking strength

can be represented by the difference between their historical average goals scored.

In [12]:
features_df["AttackDifference"] = (
    features_df["HomeAvgGoalsFor"]
    - features_df["AwayAvgGoalsFor"]
)

features_df["DefenseDifference"] = (
    features_df["AwayAvgGoalsAgainst"]
    - features_df["HomeAvgGoalsAgainst"]
)

features_df["PointsDifference"] = (
    features_df["HomePointsPerMatch"]
    - features_df["AwayPointsPerMatch"]
)

In [13]:
feature_columns = [
    "Date",
    "HomeTeam",
    "AwayTeam",
    "HomeMatchesBefore",
    "AwayMatchesBefore",
    "HomeAvgGoalsFor",
    "HomeAvgGoalsAgainst",
    "AwayAvgGoalsFor",
    "AwayAvgGoalsAgainst",
    "HomePointsPerMatch",
    "AwayPointsPerMatch",
    "AttackDifference",
    "DefenseDifference",
    "PointsDifference",
    "FTR"
]

features_df[feature_columns].head(15)

,Date,HomeTeam,AwayTeam,HomeMatchesBefore,AwayMatchesBefore,HomeAvgGoalsFor,HomeAvgGoalsAgainst,AwayAvgGoalsFor,AwayAvgGoalsAgainst,HomePointsPerMatch,AwayPointsPerMatch,AttackDifference,DefenseDifference,PointsDifference,FTR
0,2025-08-15,Liverpool,Bournemouth,0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,H
1,2025-08-16,Aston Villa,Newcastle,0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,D
2,2025-08-16,Brighton,Fulham,0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,D
3,2025-08-16,Sunderland,West Ham,0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,H
4,2025-08-16,Tottenham,Burnley,0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,H
5,2025-08-16,Wolves,Man City,0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,A
6,2025-08-17,Chelsea,Crystal Palace,0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,D
7,2025-08-17,Nott'm Forest,Brentford,0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,H
8,2025-08-17,Man United,Arsenal,0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,A
9,2025-08-18,Leeds,Everton,0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,H


## 9. Check for Data Leakage

The first match of the season should have no historical information.

Therefore, the first match should have:

- 0 previous matches for the home team
- 0 previous matches for the away team
- 0 historical goals
- 0 historical points

This is an important validation of our feature-engineering process.

In [14]:
features_df[
    [
        "Date",
        "HomeTeam",
        "AwayTeam",
        "HomeMatchesBefore",
        "AwayMatchesBefore",
        "HomeGoalsForBefore",
        "AwayGoalsForBefore"
    ]
].head()

,Date,HomeTeam,AwayTeam,HomeMatchesBefore,AwayMatchesBefore,HomeGoalsForBefore,AwayGoalsForBefore
0,2025-08-15,Liverpool,Bournemouth,0,0,0,0
1,2025-08-16,Aston Villa,Newcastle,0,0,0,0
2,2025-08-16,Brighton,Fulham,0,0,0,0
3,2025-08-16,Sunderland,West Ham,0,0,0,0
4,2025-08-16,Tottenham,Burnley,0,0,0,0


## 10. Check That Current Match Goals Were Not Used

The historical features must be based only on previous matches.

For example, `HomeGoalsForBefore` must represent goals scored before the current match, not including the current match.

This is why the statistics were recorded before updating the team records.

In [15]:
features_df.iloc[100:105][
    [
        "Date",
        "HomeTeam",
        "AwayTeam",
        "HomeMatchesBefore",
        "HomeGoalsForBefore",
        "AwayMatchesBefore",
        "AwayGoalsForBefore",
        "FTR"
    ]
]

,Date,HomeTeam,AwayTeam,HomeMatchesBefore,HomeGoalsForBefore,AwayMatchesBefore,AwayGoalsForBefore,FTR
100,2025-11-08,Tottenham,Man United,10,17,10,17,D
101,2025-11-08,Everton,Fulham,10,10,10,12,H
102,2025-11-08,West Ham,Burnley,10,10,10,12,H
103,2025-11-08,Sunderland,Arsenal,10,12,10,18,D
104,2025-11-08,Chelsea,Wolves,10,18,10,7,H


## 11. Final ML Feature Set

For the initial ML model, we can use a relatively small set of meaningful historical features.

We intentionally avoid using match statistics that are only known after the match begins.

In [16]:
ml_features = [
    "HomeAvgGoalsFor",
    "HomeAvgGoalsAgainst",
    "AwayAvgGoalsFor",
    "AwayAvgGoalsAgainst",
    "HomePointsPerMatch",
    "AwayPointsPerMatch",
    "AttackDifference",
    "DefenseDifference",
    "PointsDifference"
]

X = features_df[ml_features]

y = features_df["FTR"]

In [17]:
print("Features:")
print(X.columns.tolist())

print("\nFeature shape:", X.shape)

print("\nTarget:")
print(y.head())

Features:
['HomeAvgGoalsFor', 'HomeAvgGoalsAgainst', 'AwayAvgGoalsFor', 'AwayAvgGoalsAgainst', 'HomePointsPerMatch', 'AwayPointsPerMatch', 'AttackDifference', 'DefenseDifference', 'PointsDifference']

Feature shape: (380, 9)

Target:
0    H
1    D
2    D
3    H
4    H
Name: FTR, dtype: str


## 12. Check Missing Values

Before moving to model training, verify that the engineered features do not contain unexpected missing values.

In [18]:
X.isnull().sum()

HomeAvgGoalsFor        0
HomeAvgGoalsAgainst    0
AwayAvgGoalsFor        0
AwayAvgGoalsAgainst    0
HomePointsPerMatch     0
AwayPointsPerMatch     0
AttackDifference       0
DefenseDifference      0
PointsDifference       0
dtype: int64

## 13. Feature Summary

We inspect the statistical distribution of the engineered numerical features.

In [19]:
X.describe()

,HomeAvgGoalsFor,HomeAvgGoalsAgainst,AwayAvgGoalsFor,AwayAvgGoalsAgainst,HomePointsPerMatch,AwayPointsPerMatch,AttackDifference,DefenseDifference,PointsDifference
count,380.000000,380.000000,380.000000,380.000000,380.000000,380.000000,380.000000,380.000000,380.000000
mean,1.310182,1.335370,1.334839,1.310151,1.317930,1.361170,-0.024657,-0.025219,-0.043241
std,0.510417,0.530173,0.517800,0.527570,0.570760,0.580423,0.645878,0.694476,0.728932
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,-4.000000,-3.000000,-3.000000
25%,1.048864,1.051378,1.052632,1.000000,1.000000,1.058007,-0.372321,-0.416667,-0.526786
50%,1.333333,1.347826,1.338095,1.333333,1.375000,1.387993,0.000000,0.000000,0.000000
75%,1.603226,1.625000,1.620536,1.580044,1.655172,1.642857,0.371528,0.400000,0.445772
max,4.000000,4.000000,4.000000,4.000000,3.000000,3.000000,2.000000,3.000000,2.066667


## 14. Save the Engineered Dataset

The engineered dataset will be saved separately from the cleaned dataset.

The raw dataset and cleaned dataset remain unchanged.

In [20]:
features_df.to_csv(
    "../data/processed/PremierLeague25_26_features.csv",
    index=False
)

print("Feature-engineered dataset saved successfully.")

Feature-engineered dataset saved successfully.


In [21]:
feature_check = pd.read_csv(
    "../data/processed/PremierLeague25_26_features.csv"
)

print("Shape:", feature_check.shape)

feature_check.head()

Shape: (380, 21)


,Date,HomeTeam,AwayTeam,HomeMatchesBefore,HomeGoalsForBefore,HomeGoalsAgainstBefore,HomePointsBefore,AwayMatchesBefore,AwayGoalsForBefore,AwayGoalsAgainstBefore,...,FTR,HomeAvgGoalsFor,HomeAvgGoalsAgainst,AwayAvgGoalsFor,AwayAvgGoalsAgainst,HomePointsPerMatch,AwayPointsPerMatch,AttackDifference,DefenseDifference,PointsDifference
0,2025-08-15,Liverpool,Bournemouth,0,0,0,0,0,0,0,...,H,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,2025-08-16,Aston Villa,Newcastle,0,0,0,0,0,0,0,...,D,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,2025-08-16,Brighton,Fulham,0,0,0,0,0,0,0,...,D,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,2025-08-16,Sunderland,West Ham,0,0,0,0,0,0,0,...,H,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,2025-08-16,Tottenham,Burnley,0,0,0,0,0,0,0,...,H,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
